In [ ]:
# Import Libraries
import requests

import pandas as pd
import numpy as np
import geopandas as gpd
from shapely.geometry import Point

import os
from os import path
import json
import time
import random
from pathlib import Path
from dotenv import load_dotenv
from scipy.spatial import cKDTree


In [21]:
# Get file names 
hdb_points_file = "../datasets/resale_2019_to_2025_cleaned.geojson"
malls_points_file = "../datasets/singapore_malls_pois.csv"
output_file = "../cleaned_datasets/hdb_mall_distances.csv"

In [15]:
load_dotenv()

url = "https://www.onemap.gov.sg/api/auth/post/getToken"

payload = {
    "email": os.getenv("ONEMAP_EMAIL"),
    "password": os.getenv("ONEMAP_EMAIL_PASSWORD")
}

response = requests.post(url, json=payload)

if response.status_code == 200:
    data = response.json()
    onemap_api_key = data.get("access_token")  
else:
    onemap_api_key = None
    print("Failed to get token")

### 1. Get names and coordinates of shopping malls

In [11]:
# read file 
malls_df = pd.read_csv(malls_points_file)

# filter out rows where 'name' is null and select only the relevant columns
malls_df = malls_df[malls_df['name'].notna()][['name', 'lat', 'lon']].copy()

# remove duplicates based on 'name' and keep the first occurrence
malls_df = malls_df.drop_duplicates(subset='name', keep='first')

print(f"Remaining malls: {len(malls_df)}")
print(malls_df.head())

Remaining malls: 219
                         name       lat         lon
0              The Star Vista  1.308002  103.788382
1  Bencoolen Underground Mall  1.298181  103.849647
3                    Katong V  1.303133  103.903231
4             The Poiz Centre  1.331436  103.868571
5         Clarke Quay Central  1.288999  103.846180


In [ ]:
# convert malls_df to a GeoDataFrame
malls_gdf = gpd.GeoDataFrame(
    malls_df,
    geometry=gpd.points_from_xy(malls_df['lon'], malls_df['lat']),
    crs="EPSG:4326"
)
# reproject to a projected CRS for accurate distance calculations 
malls_gdf = malls_gdf.to_crs(epsg=3414)
malls_gdf.head()

,name,lat,lon,geometry
0,The Star Vista,1.308002,103.788382,POINT (22999.033 32257.723)
1,Bencoolen Underground Mall,1.298181,103.849647,POINT (29817.23 31171.828)
3,Katong V,1.303133,103.903231,POINT (35780.64 31719.497)
4,The Poiz Centre,1.331436,103.868571,POINT (31923.218 34848.973)
5,Clarke Quay Central,1.288999,103.846180,POINT (29431.322 30156.516)


### 2. Get walking distance between HDB and closest mall 

3.1 Run a sample of postal codes to ensure code works, check the result output to see if there are any nulls 

In [16]:
# --- 1. load and prep data ---
raw_hdb_gdf = gpd.read_file(hdb_points_file).to_crs(epsg=4326)
# take only the first 5 unique postals for this sample
postal_gdf_sample = raw_hdb_gdf.drop_duplicates(subset=['postal_code']).head(5).copy()

malls_gdf = malls_gdf.to_crs(epsg=4326)

mall_coords = list(zip(malls_gdf.geometry.y, malls_gdf.geometry.x))
mall_names = malls_gdf['name'].tolist()

tree = cKDTree(mall_coords)

results = []

print(f"--- starting sample test for {len(postal_gdf_sample)} postals ---")

# --- 2. sample loop ---
for idx, row in postal_gdf_sample.iterrows():
    p_code = str(row['postal_code'])
    origin_coords = (row.geometry.y, row.geometry.x)
    
    print(f"\nchecking postal: {p_code} at {origin_coords}")
    
    # get 3 closest malls by straight line
    dist, indices = tree.query(origin_coords, k=3) 
    
    best_walk = float('inf')
    closest_mall = None

    for i in indices:
        dest_coords = mall_coords[i]
        mall_name = mall_names[i]
        
        # using the routingsvc endpoint from the docs
        url = f"https://www.onemap.gov.sg/api/public/routingsvc/route?start={origin_coords[0]},{origin_coords[1]}&end={dest_coords[0]},{dest_coords[1]}&routeType=walk"
        
        try:

            token = onemap_api_key if onemap_api_key.startswith("Bearer ") else f"Bearer {onemap_api_key}"
            headers = {"Authorization": token}
            # headers = {"Authorization": onemap_api_key}
            res = requests.get(url, headers=headers, timeout=10)
            
            if res.status_code == 200:
                data = res.json()
                if 'route_summary' in data:
                    walk_m = data['route_summary']['total_distance']
                    print(f"  -> found route to {mall_name}: {walk_m}m")
                    if walk_m < best_walk:
                        best_walk = walk_m
                        closest_mall = mall_name
                else:
                    print(f"  -> no route summary found for {mall_name}")
            else:
                print(f"  -> api error {res.status_code}: {res.text}")
            
            time.sleep(0.2) # slightly longer sleep for the sample
        except Exception as e:
            print(f"  -> request failed: {e}")

    results.append({
        "postal_code": p_code,
        "nearest_mall": closest_mall,
        "walking_dist_m": best_walk if best_walk != float('inf') else None
    })

# --- 3. display output ---
sample_df = pd.DataFrame(results)
print("\n--- final sample output ---")
print(sample_df)

# check if we actually got results
if sample_df['walking_dist_m'].isnull().all():
    print("\nwarning: all walking distances are null. check your api key and coordinate order.")
else:
    print("\nsuccess: distances found! you can now run the full script.")

--- starting sample test for 5 postals ---

checking postal: 560225 at (1.3673961277036402, 103.83815000747873)
  -> found route to Broadway Plaza: 1424m
  -> found route to Ang Mo Kio Hub: 1437m
  -> found route to Jubilee Square: 1634m

checking postal: 560174 at (1.3751965588189876, 103.83769584780252)
  -> found route to Broadway Plaza: 1461m
  -> found route to Jubilee Square: 1671m
  -> found route to Ang Mo Kio Hub: 1874m

checking postal: 560440 at (1.3664278983686127, 103.85431149122876)
  -> found route to Ang Mo Kio Hub: 1059m
  -> found route to Jubilee Square: 1280m
  -> found route to Broadway Plaza: 1447m

checking postal: 560637 at (1.3803292495669066, 103.8423366937204)
  -> found route to Broadway Plaza: 1281m
  -> found route to Jubilee Square: 1464m
  -> found route to Ang Mo Kio Hub: 1777m

checking postal: 560571 at (1.3701655201821288, 103.85494058284989)
  -> found route to Ang Mo Kio Hub: 899m
  -> found route to Jubilee Square: 1104m
  -> found route to Broadw

3.2 Run full script if results generated for the sample postal codes

In [32]:
# --- 1. Load Datasets (full) ---
raw_hdb_gdf = gpd.read_file(hdb_points_file).to_crs(epsg=4326)
postal_gdf = raw_hdb_gdf.drop_duplicates(subset=['postal_code']).copy()

malls_gdf = malls_gdf.to_crs(epsg=4326)
mall_coords = list(zip(malls_gdf.geometry.y, malls_gdf.geometry.x))
mall_names = malls_gdf['name'].tolist()
tree = cKDTree(mall_coords)

# --- 2. Resume logic ---
if os.path.exists(output_file):
    processed_df = pd.read_csv(output_file)
    processed_postals = set(processed_df['postal_code'].astype(str))
    results = processed_df.to_dict('records')
    print(f"Resuming... {len(processed_postals)} postals already completed.")
else:
    results = []
    processed_postals = set()

# --- 3. Full loop ---
print(f"Starting processing for {len(postal_gdf) - len(processed_postals)} remaining postals...")

try:
    for idx, row in postal_gdf.iterrows():
        p_code = row['postal_code']
        if p_code in processed_postals:
            continue

        origin_coords = (row.geometry.y, row.geometry.x)
        dist, indices = tree.query(origin_coords, k=3)
        
        closest_walk = float('inf')
        closest_mall = None

        for i in indices:
            dest_coords = mall_coords[i]
            token = onemap_api_key if onemap_api_key.startswith("Bearer ") else f"Bearer {onemap_api_key}"
            url = f"https://www.onemap.gov.sg/api/public/routingsvc/route?start={origin_coords[0]},{origin_coords[1]}&end={dest_coords[0]},{dest_coords[1]}&routeType=walk"
            
            try:
                res = requests.get(url, headers={"Authorization": token}, timeout=10)
                if res.status_code == 200:
                    data = res.json()
                    if 'route_summary' in data:
                        walk_m = data['route_summary']['total_distance']
                        if walk_m < closest_walk:
                            closest_walk = walk_m
                            closest_mall = mall_names[i]
                    else:
                        print(f"No route_summary for postal {p_code} -> mall {mall_names[i]}")
                else:
                    print(f"API error for postal {p_code} -> mall {mall_names[i]}: {res.status_code}")
                    print(res.text[:300])
                time.sleep(0.2)
            except:
                continue

        results.append({
            "postal_code": p_code,
            "nearest_mall": closest_mall,
            "walking_dist_m": closest_walk if closest_walk != float('inf') else None
        })
        processed_postals.add(p_code)

        if len(results) % 50 == 0:
            pd.DataFrame(results).to_csv(output_file, index=False)
            print(f"progress: {len(results)} / {len(postal_gdf)} unique postals saved.")
            

except KeyboardInterrupt:
    print("\nPaused. Saving current progress...")

# Final save
pd.DataFrame(results).to_csv(output_file, index=False)
print(f"Task complete! Results saved to {output_file}")

Starting processing for 9650 remaining postals...
progress: 50 / 9650 unique postals saved.
progress: 100 / 9650 unique postals saved.
progress: 150 / 9650 unique postals saved.
progress: 200 / 9650 unique postals saved.
progress: 250 / 9650 unique postals saved.
progress: 300 / 9650 unique postals saved.
progress: 350 / 9650 unique postals saved.
progress: 400 / 9650 unique postals saved.
progress: 450 / 9650 unique postals saved.
progress: 500 / 9650 unique postals saved.
progress: 550 / 9650 unique postals saved.
progress: 600 / 9650 unique postals saved.
progress: 650 / 9650 unique postals saved.
progress: 700 / 9650 unique postals saved.
progress: 750 / 9650 unique postals saved.
progress: 800 / 9650 unique postals saved.
progress: 850 / 9650 unique postals saved.
progress: 900 / 9650 unique postals saved.
progress: 950 / 9650 unique postals saved.
progress: 1000 / 9650 unique postals saved.
progress: 1050 / 9650 unique postals saved.
progress: 1100 / 9650 unique postals saved.
pr

In [35]:
mall_distances = pd.read_csv("../cleaned_datasets/hdb_mall_distances.csv")
mall_distances.info()

<class 'pandas.DataFrame'>
RangeIndex: 9650 entries, 0 to 9649
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   postal_code     9650 non-null   int64
 1   nearest_mall    9650 non-null   str  
 2   walking_dist_m  9650 non-null   int64
dtypes: int64(2), str(1)
memory usage: 226.3 KB


### 5. Get feature variables of malls 
- num_malls_1km
- num_malls_2km

In [41]:
# read files
unique_hdb = gpd.read_file(hdb_points_file).copy()
# malls_gdf -> malls dataframe
hdb_mall_distances = pd.read_csv("../datasets/hdb_mall_distances.csv")

# get unique hdb locations
unique_hdb = unique_hdb.drop_duplicates(subset=["postal_code"]).copy()
unique_hdb = unique_hdb.to_crs(epsg=3414)
malls_gdf = malls_gdf.to_crs(epsg=3414)

# prepare unique hdb and mall coordinates for distance calculation
hdb_coords = np.array([(geom.x, geom.y) for geom in unique_hdb.geometry])
mall_coords = np.array([(geom.x, geom.y) for geom in malls_gdf.geometry])

mall_names = malls_gdf["name"].tolist()

# build cKDTree for efficient spatial queries
tree = cKDTree(mall_coords)

# get malls within 500m and 1km
indices_1km = tree.query_ball_point(hdb_coords, r=1000)
indices_2km = tree.query_ball_point(hdb_coords, r=2000)

# convert index arrays into mall name lists
unique_hdb["malls_within_1km"] = [
    [mall_names[i] for i in idx_list] for idx_list in indices_1km]

unique_hdb["malls_within_2km"] = [
    [mall_names[i] for i in idx_list] for idx_list in indices_2km]

# count columns
unique_hdb["num_malls_1km"] = unique_hdb["malls_within_1km"].apply(len)
unique_hdb["num_malls_2km"] = unique_hdb["malls_within_2km"].apply(len)

malls_counts = unique_hdb[[
    "postal_code",
    "malls_within_1km",
    "malls_within_2km",
    "num_malls_1km",
    "num_malls_2km"
]].copy()

print(malls_counts)

       postal_code                  malls_within_1km  \
0           560225                                []   
1           560174                  [Broadway Plaza]   
2           560440  [Jubilee Square, Ang Mo Kio Hub]   
4           560637                                []   
5           560571  [Jubilee Square, Ang Mo Kio Hub]   
...            ...                               ...   
178362      763478                                []   
178365      761478                                []   
179299      761795                   [Wisteria Mall]   
179458      762479                   [Junction Nine]   
179459      763479                                []   

                                         malls_within_2km  num_malls_1km  \
0       [Broadway Plaza, Jubilee Square, Thomson Plaza...              0   
1        [Broadway Plaza, Jubilee Square, Ang Mo Kio Hub]              1   
2       [Broadway Plaza, Jubilee Square, Ang Mo Kio Hu...              2   
4        [Broadway Plaz

### 6. Merge with mall distance and conduct data integrity checks

In [27]:
hdb_mall_distances = pd.read_csv("../cleaned_datasets/hdb_mall_distances.csv")
print(hdb_mall_distances.head())

   postal_code    nearest_mall  walking_dist_m
0       560225  Broadway Plaza          1424.0
1       560174  Broadway Plaza          1461.0
2       560440  Ang Mo Kio Hub          1059.0
3       560637  Broadway Plaza          1281.0
4       560571  Ang Mo Kio Hub           899.0


In [37]:
# convert postal_code to string for merging
hdb_mall_distances["postal_code"] = hdb_mall_distances["postal_code"].astype(str).str.zfill(6)
malls_counts["postal_code"] = malls_counts["postal_code"].astype(str).str.zfill(6)
hdb_mall_distances["nearest_mall"] = hdb_mall_distances["nearest_mall"].astype(str)

hdb_mall_final = hdb_mall_distances.merge(
    malls_counts,
    on="postal_code",
    how="left"
)

hdb_mall_final = hdb_mall_final[[
    "postal_code",
    "walking_dist_m",
    "num_malls_1km",
    "num_malls_2km"
]]

print(hdb_mall_final.head())
print(hdb_mall_final.info())

  postal_code  walking_dist_m  num_malls_1km  num_malls_2km
0      560225            1424              0              4
1      560174            1461              1              3
2      560440            1059              2              5
3      560637            1281              0              3
4      560571             899              2              4
<class 'pandas.DataFrame'>
RangeIndex: 9650 entries, 0 to 9649
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   postal_code     9650 non-null   str  
 1   walking_dist_m  9650 non-null   int64
 2   num_malls_1km   9650 non-null   int64
 3   num_malls_2km   9650 non-null   int64
dtypes: int64(3), str(1)
memory usage: 301.7 KB
None


In [31]:
print(hdb_mall_distances['walking_dist_m'].isna())

0       False
1       False
2       False
3       False
4       False
        ...  
9645     True
9646     True
9647     True
9648     True
9649     True
Name: walking_dist_m, Length: 9650, dtype: bool


In [30]:
# Save file to csv 
hdb_mall_final.to_csv("../cleaned_datasets/hdb_mall_distances_final.csv", index=False)